# Portadas infantiles generativas — etapas 1 a 3
Cuaderno reproducible de las etapas 2 (datos) y 3 (modelo generativo) del Proyecto 2.
**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU T4*.

Orden: (0) preparación · (1) pruebas · (2) datos y análisis exploratorio [Jimena] · (3) motores, calibración, E1 y E5 [Maria].

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 0. Preparación
Sube la carpeta `codigo/` a tu Google Drive (por ejemplo en `MiUnidad/planif_proyecto/codigo`) y ajusta la ruta.
Todo es gratuito: los dos motores y los modelos se descargan de Hugging Face y corren en la GPU de Colab.
Con Colab Pro conviene elegir una GPU L4 o A100: las evaluaciones largas terminan mucho antes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/planif_proyecto/codigo

In [ ]:
!pip install -q -r requirements.txt

## 1. Pruebas (CPU, sin modelos)

In [ ]:
!python -m pytest -q tests

## 2. Datos (etapa 2)
- **D1** 2 000 portadas de *Children's Books* (solo estadísticas; no se muestran ni redistribuyen).
- **D2** 200 bocetos de Quick, Draw! (CC BY 4.0).
- **D4** 200 historias de TinyStories (CDLA-Sharing-1.0) en inglés + plantilla para anotar 30.
- Análisis exploratorio con OCR de la posición del título.

Para una prueba rápida usa `--n-d1 200`.

**Opción A (recomendada):** los datos ya se descargaron en una computadora y la carpeta `datos/` se subió dentro de `codigo/` en Drive. Solo se hace el análisis exploratorio con OCR (las historias se usan en inglés, sin traducir).

In [ ]:
!ls datos && python -m scripts.preparar_datos --solo eda --ocr

**Opción A' (Kaggle):** si subiste `datos.zip` como dataset **privado** de Kaggle, guarda `KAGGLE_USERNAME` y `KAGGLE_KEY` en *Secrets* y cambia el nombre del dataset.

In [ ]:
# import os, shutil, kagglehub
# from google.colab import userdata
# os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME'); os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
# ruta = kagglehub.dataset_download('TU_USUARIO/portadas-infantiles-datos')
# shutil.copytree(f'{ruta}/datos', 'datos', dirs_exist_ok=True)
# !python -m scripts.preparar_datos --solo eda --ocr

**Opción B:** descargar todo desde Colab (puede fallar si Amazon bloquea las IPs de Colab).

In [ ]:
# !python -m scripts.preparar_datos --ocr

In [ ]:
import json
from IPython.display import Image as Img, display
print(json.dumps(json.load(open('salidas/eda/d1_descarga.json')), indent=1))
r = json.load(open('salidas/eda/resumen_d1.json'))
print('n =', r['n'], '| posición del título:', r.get('tercio_titulo'))
print('umbral densidad zona del título (p75):', round(r['umbral_densidad_titulo'], 4))
for k in ['colorido', 'brillo', 'saturacion', 'dens_sup', 'dens_inf']:
    print(k, {a: round(b, 3) for a, b in r['metricas'][k].items()})
display(Img('salidas/eda/d1_distribuciones.png'))
display(Img('salidas/eda/d1_densidad_tercios.png'))

## 3.1 Motores (primera ronda de 5 portadas)
Motor A = SD 1.5 + ControlNet (nivel intermedio). Motor B = SD-Turbo img2img sin control (nivel básico, respaldo y línea base).

In [ ]:
!python -m scripts.demo_ronda --quickdraw rabbit --escena "a rabbit among flowers in a garden" --eje P1 --motor A
display(Img('salidas/demo/lamina.png'))

In [ ]:
# Con tu propio boceto: súbelo al panel de archivos y cambia la ruta
# !python -m scripts.demo_ronda --boceto mi_boceto.jpg --escena "a rabbit among flowers" --eje P2

In [ ]:
# Motor B: SD-Turbo (4 pasos, mucho más rápido)
!python -m scripts.demo_ronda --quickdraw rabbit --escena "a rabbit among flowers in a garden" --eje P1 --motor B --salida salidas/demo_b
display(Img('salidas/demo_b/lamina.png'))

## 3.2 Calibración de la fidelidad y los pasos
360 imágenes (20 bocetos × 6 escalas × 3 pasos), unos 30–40 min en una T4 (menos en L4/A100).
Al terminar, la celda siguiente copia la propuesta a `portadas/config.py` (`FIDELIDAD` y `PASOS`) y corre las pruebas.

In [ ]:
!python -m experimentos.calibracion --n 20
print(open('salidas/calibracion/resumen.json').read()[-600:])
display(Img('salidas/calibracion/calibracion.png'))

In [ ]:
# Copia los valores propuestos a config.py (guarda config.py.antes_calibracion)
!python -m experimentos.aplicar_calibracion && python -m pytest -q tests

## 3.3 Experimento opcional con LoRA
Compara el motor A con y sin un LoRA de ilustración infantil (mismos bocetos, prompts y semillas). Primero se revisa la ficha del LoRA: licencia, modelo base SD 1.5 y que no imite a un artista. Por defecto usa `alvdansen/dimension-w-sd15` (OpenRAIL-M); su ficha no documenta los datos de entrenamiento, lo que se reporta como limitación. ~20 min en T4.

In [ ]:
!python -m experimentos.lora --revisar

In [ ]:
!python -m experimentos.lora --n 20
print(open('salidas/lora/resumen.json').read()[-900:])
display(Img('salidas/lora/lamina_lora.png'))

## E1. Evaluación automática
Primero una prueba con 5 pares; luego la completa (200 pares: 2 000 portadas del motor A y 1 000 del motor B).
Si Colab se desconecta, vuelve a correr la misma celda: continúa desde `progreso.jsonl`.

In [ ]:
!python -m experimentos.e1_automatica --pares 5 --pares-b 5 --salida salidas/e1_prueba --sin-kid

In [ ]:
!python -m experimentos.e1_automatica --pares 200 --pares-b 200

## E5. Seguridad y sesgo

In [ ]:
!python -m experimentos.e5_seguridad --parte a

In [ ]:
!python -m experimentos.e5_seguridad --parte b

**Pendiente manual:** completar `salidas/e5/e5a_revision_manual.csv` (¿cada portada mostrada es apta?) y
`salidas/e5/e5b_anotacion_sesgo.csv` (género y tono de piel percibidos), idealmente con dos anotadoras.